In [1]:
import random,os
import pandas as pd
import numpy as np
import missingno as msno
from Data_cleaning_utils import perform_data_cleaning
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder,StandardScaler
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,KNNImputer,IterativeImputer
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_validate, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from catboost import CatBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn import set_config
set_config(transform_output='pandas')
import optuna as optuna


def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed = 0
seed_everything(seed)

import dagshub
dagshub.init(repo_owner='shapniljoy', repo_name='delivery-time-prediction', mlflow=True)

import mlflow
mlflow.set_tracking_uri("https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow")
mlflow.set_experiment('StackingRegressor HP Tuning')

Accessing as shapniljoy

Initialized MLflow to track repo "shapniljoy/delivery-time-prediction"

Repository shapniljoy/delivery-time-prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/751c2160b14b43f4a222918cbcadbffb', creation_time=1782111186062, experiment_id='6', last_update_time=1782111186062, lifecycle_stage='active', name='StackingRegressor HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

Previously we have run 3 different models, 2 boosting algorithms CatboostRegressor and LightGBMRegressor and 1 bagging algorithm Random forest.  
The final Cross validation scores after dropping missing values and With Numerical columns are,  
(1) LightGBMRegressor,  
Training cv mae : 2.45 , Test cv mae : 3.09  
(2) CatboostRegressor,  
Training cv mae : 2.60 , Test cv mae : 3.13  
(3) RandomForestRegressor,  
Training cv mae : 2.54 , Test cv mae : 3.08  


Although all of the 3 models are slightly overfitting in the data and showing almost simillar type of mae cv score, we shall choose the models with lower gap in the training cv and test cv.
So we choose CatboostRegressor and RandomForestRegressor for further StackingRegressor method.  
It is also adviseble to use 2 different type of algorithms in stacking regressor to get mor robust results and hyperparameter tuning.

In [8]:
data = pd.read_csv(r"C:/Users/User/delivery-time-prediction/data/raw/swiggy.csv")
df = perform_data_cleaning(data)
df = df.dropna()

df = df.drop(columns=['rider_id','restaurant_lat', 'restaurant_long',
       'delivery_lat', 'delivery_long', 'city_name', 'order_time_hour','age',
        'ratings','distance_km','pickup_time','day','month'])
x = df.drop(columns=['time'])
y = df['time'] 

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=seed)

print(f"Training data: {x_train.shape} and {y_train.shape}", '\n')
print(f"Testing data: {x_test.shape} and {y_test.shape}", '\n')

Training data: (30156, 15) and (30156,) 

Testing data: (7539, 15) and (7539,) 



In [9]:
rating_order = ['less than 4','4-4.5','4.5-5']

distance_order = ['short','medium','long','very_long']

city_type_order = ['Semi-Urban','Urban','Metropolitan']

pickup_time_order = ['5 minutes','10 minutes','15 minutes']

ordinal_encoding = OrdinalEncoder(categories=[rating_order,distance_order,city_type_order,pickup_time_order],
                                  handle_unknown='use_encoded_value', unknown_value=-999)


preprocessor = ColumnTransformer([
        ('OE',ordinal_encoding, ['ratings_cat','distance_km_cat','city_type','pickup_time_cat']),
        ('OHE', OneHotEncoder(drop='first',sparse_output=False), ['vehicle_type','order_type','day_of_week',
                                                                  'age_cat','weather','traffic','festival','time_of_day']),
        ],remainder='passthrough')

In [10]:
lgbm_params = {
 'n_estimators': 1940,
 'max_depth': 7,
 'num_leaves': 57,
 'min_child_samples': 228,
 'colsample_bytree': 0.8452545147323064,
 'learning_rate': 0.046640660322683554,
 'reg_alpha': 0.020341454480246005,
 'reg_lambda': 0.10776961230711216,
 'boosting_type': 'dart',
 'subsample': 0.9481361810641186
}

RF_params = {
    'n_estimators': 641,
    'max_depth': 13,
    'min_samples_split': 6,
    'min_samples_leaf': 6,
    'max_features': None,
    'max_samples': 0.992036436296416
}

x_train_preprocessed = preprocessor.fit_transform(x_train)

lgbm = LGBMRegressor(**lgbm_params,verbose=-1)
lgbm_predicted = cross_val_predict(lgbm, x_train_preprocessed, y_train,
                                  cv=KFold(n_splits=5,shuffle=True,random_state=seed),n_jobs=-1,verbose=False)

rf = RandomForestRegressor(**RF_params)
rf_predicted = cross_val_predict(rf, x_train_preprocessed, y_train,
                                cv=KFold(n_splits=5,shuffle=True,random_state=seed),n_jobs=-1,verbose=False)

In [13]:
meta_predicted = np.stack((lgbm_predicted,rf_predicted),axis=1)
meta_predicted

array([[21.65329502, 22.90266685],
       [16.93845254, 16.6351303 ],
       [17.91516069, 19.7005672 ],
       ...,
       [24.52027423, 24.53455992],
       [26.77802184, 25.81897452],
       [31.11906291, 33.53336038]], shape=(30156, 2))

In [18]:
def objective(trial):

    with mlflow.start_run(nested=True,run_name=f"trial_{trial.number}"):

        stacking_models = trial.suggest_categorical('stacking_models', ['LR','Lasso','Ridge','DT'])

        if stacking_models == 'LR':
            estimator = LinearRegression()
            
        elif stacking_models == 'DT':

            max_depth = trial.suggest_int('max_depth', 4, 10)
            max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 5, 10)
            min_samples_split = trial.suggest_int('min_samples_split', 5, 20)
            min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 10)
            max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

            estimator = DecisionTreeRegressor(max_depth=max_depth, max_leaf_nodes=max_leaf_nodes,
                                            min_samples_split=min_samples_split,
                                            min_samples_leaf=min_samples_leaf,
                                            max_features=max_features, random_state=seed)
        
        elif stacking_models == 'Lasso':

            alpha = trial.suggest_float('alpha', 0.001,0.5, log=True)

            estimator = Lasso(alpha=alpha ,random_state=seed)

        elif stacking_models == 'Ridge':
            alpha = trial.suggest_float('alpha', 0.001,0.5, log=True)
            solver = trial.suggest_categorical('solver', ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga'])

            estimator = Ridge(alpha=alpha,solver=solver ,random_state=seed)
        
        
        # model = Pipeline([
            # ('preprocessor',preprocessor),
            # ('Stacking_estimator', estimator)
        # ])  

        cv_score = cross_validate(estimator, meta_predicted, y_train, cv=KFold(n_splits=5,shuffle=True,random_state=seed), 
                                scoring='neg_mean_absolute_error',verbose=False,n_jobs=-1,return_train_score=True)
        
        trial.set_user_attr('train_cv',np.mean(cv_score['train_score']))
        trial.set_user_attr('val_cv',np.mean(cv_score['test_score']))
        
        mlflow.log_params(trial.params)
        mlflow.log_metric('train_cv_score',np.mean(cv_score['train_score']))
        mlflow.log_metric('val_cv_score',np.mean(cv_score['test_score']))
            
        return np.mean(cv_score['test_score'])


study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=seed),
                            study_name='StackingRegressor Hyperparameter Tuning')
    
with mlflow.start_run(run_name='StackingRegressor') as parent :
    study.optimize(objective,n_trials=30,show_progress_bar=True)

    mlflow.log_metric('StackingRegressor best Score',study.best_value)
    mlflow.log_params(study.best_params)

    print('Best parameters:', study.best_params)
    print('Best score:', study.best_value, '\n')

[I 2026-06-22 13:40:21,348] A new study created in memory with name: StackingRegressor Hyperparameter Tuning


  0%|          | 0/30 [00:00<?, ?it/s]

🏃 View run trial_0 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/6/runs/672589ba77664e98a3c64d5c2684d856
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/6
[I 2026-06-22 13:40:27,840] Trial 0 finished with value: -3.5529375168940254 and parameters: {'stacking_models': 'Lasso', 'alpha': 0.013913346327363716}. Best is trial 0 with value: -3.5529375168940254.
🏃 View run trial_1 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/6/runs/eb28e14ca092485982d8511764e066ac
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/6
[I 2026-06-22 13:40:32,414] Trial 1 finished with value: -3.69256445318195 and parameters: {'stacking_models': 'DT', 'max_depth': 6, 'max_leaf_nodes': 9, 'min_samples_split': 13, 'min_samples_leaf': 8, 'max_features': 'sqrt'}. Best is trial 0 with value: -3.5529375168940254.
🏃 View run trial_2 at: https://dagshub.

In [21]:
study.trials_dataframe()[['user_attrs_train_cv','user_attrs_val_cv','params_stacking_models']].sort_values(by='user_attrs_val_cv',ascending=False)

,user_attrs_train_cv,user_attrs_val_cv,params_stacking_models
29,-3.552161,-3.552611,Lasso
25,-3.552166,-3.552616,Lasso
28,-3.552188,-3.552638,Lasso
27,-3.552188,-3.552639,Lasso
22,-3.552236,-3.552688,Lasso
24,-3.552252,-3.552704,Lasso
21,-3.552253,-3.552705,Lasso
20,-3.552268,-3.552720,Lasso
19,-3.552307,-3.552760,Lasso
15,-3.552444,-3.552896,Ridge


In [22]:
study.best_params

{'stacking_models': 'Lasso', 'alpha': 0.2606044403283886}

# Optuna after dropping missing values and With Numerical columns

In [2]:
data = pd.read_csv(r"C:/Users/User/delivery-time-prediction/data/raw/swiggy.csv")
df = perform_data_cleaning(data)
df = df.dropna()

df = df.drop(columns=['rider_id','restaurant_lat', 'restaurant_long',
       'delivery_lat', 'delivery_long', 'city_name', 'order_time_hour','day','month',
        'age_cat','ratings_cat','distance_km_cat','pickup_time_cat'])
x = df.drop(columns=['time'])
y = df['time'] 

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=seed)

print(f"Training data: {x_train.shape} and {y_train.shape}", '\n')
print(f"Testing data: {x_test.shape} and {y_test.shape}", '\n')

Training data: (30156, 15) and (30156,) 

Testing data: (7539, 15) and (7539,) 



In [3]:

numerical_cols = ['age','ratings','distance_km','pickup_time']

city_type_order = ['Semi-Urban','Urban','Metropolitan']

ordinal_encoding = OrdinalEncoder(categories=[city_type_order],
                                  handle_unknown='use_encoded_value', unknown_value=-999)


preprocessor = ColumnTransformer([
        ('OE',ordinal_encoding, ['city_type']),
        ('OHE', OneHotEncoder(drop='first',sparse_output=False), ['vehicle_type','order_type','day_of_week',
                                                                  'weather','traffic','festival','time_of_day']),
        ('scaling',StandardScaler(),numerical_cols),
        ],remainder='passthrough')

In [6]:
catboost_params = {'iterations': 618, 
                  'depth': 7, 
                  'learning_rate': 0.029550509331092316, 
                  'l2_leaf_reg': 31.153463513382516, 
                  'random_strength': 1, 
                  'bagging_temperature': 0.7163272041185655, 
                  'border_count': 96, 
                  'grow_policy': 'Depthwise', 
                  'max_ctr_complexity': 7}

RF_params = {'n_estimators': 902,
            'max_depth': 14,
            'min_samples_split': 10,
            'min_samples_leaf': 7,
            'max_features': None,
            'max_samples': 0.9857978714826628}

x_train_preprocessed = preprocessor.fit_transform(x_train)

catboost_model = CatBoostRegressor(**catboost_params,random_state=seed)
catboost_predicted = cross_val_predict(catboost_model,x_train_preprocessed,y_train,cv=KFold(n_splits=5,shuffle=True,random_state=seed))

rf_model = RandomForestRegressor(**RF_params,bootstrap=True,random_state=seed,verbose=False)
rf_predicted = cross_val_predict(rf_model,x_train_preprocessed,y_train,cv=KFold(n_splits=5,shuffle=True,random_state=seed))

x_predicted = np.stack([catboost_predicted,rf_predicted],axis=1)
x_predicted 

0:	learn: 9.1994034	total: 6.92ms	remaining: 4.27s
1:	learn: 9.0383543	total: 13.5ms	remaining: 4.15s
2:	learn: 8.8863865	total: 20.5ms	remaining: 4.2s
3:	learn: 8.7417051	total: 27.2ms	remaining: 4.18s
4:	learn: 8.5999262	total: 34.6ms	remaining: 4.25s
5:	learn: 8.4591826	total: 40.8ms	remaining: 4.17s
6:	learn: 8.3208659	total: 49.5ms	remaining: 4.32s
7:	learn: 8.1930455	total: 56.3ms	remaining: 4.29s
8:	learn: 8.0687367	total: 63.2ms	remaining: 4.27s
9:	learn: 7.9486193	total: 70.5ms	remaining: 4.29s
10:	learn: 7.8329411	total: 77.2ms	remaining: 4.26s
11:	learn: 7.7189044	total: 83.7ms	remaining: 4.23s
12:	learn: 7.6171802	total: 89.5ms	remaining: 4.16s
13:	learn: 7.5214851	total: 96.3ms	remaining: 4.16s
14:	learn: 7.4245420	total: 103ms	remaining: 4.12s
15:	learn: 7.3280866	total: 109ms	remaining: 4.12s
16:	learn: 7.2364880	total: 116ms	remaining: 4.08s
17:	learn: 7.1486789	total: 122ms	remaining: 4.08s
18:	learn: 7.0652945	total: 129ms	remaining: 4.05s
19:	learn: 6.9836994	total: 

array([[18.6514567 , 20.43345701],
       [17.87152462, 19.03858179],
       [18.39610077, 19.81176165],
       ...,
       [24.89918091, 23.6647663 ],
       [23.61400121, 22.04573985],
       [33.28339336, 32.96339901]], shape=(30156, 2))

In [7]:
def objective(trial):

    with mlflow.start_run(nested=True,run_name=f"trial_{trial.number}"):

        stacking_models = trial.suggest_categorical('stacking_models', ['LR','Lasso','Ridge','DT'])

        if stacking_models == 'LR':
            estimator = LinearRegression()
            
        elif stacking_models == 'DT':

            max_depth = trial.suggest_int('max_depth', 4, 10)
            max_leaf_nodes = trial.suggest_int('max_leaf_nodes', 5, 10)
            min_samples_split = trial.suggest_int('min_samples_split', 5, 20)
            min_samples_leaf = trial.suggest_int('min_samples_leaf', 5, 10)
            max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

            estimator = DecisionTreeRegressor(max_depth=max_depth, max_leaf_nodes=max_leaf_nodes,
                                            min_samples_split=min_samples_split,
                                            min_samples_leaf=min_samples_leaf,
                                            max_features=max_features, random_state=seed)
        
        elif stacking_models == 'Lasso':

            alpha = trial.suggest_float('alpha', 0.001,0.5, log=True)

            estimator = Lasso(alpha=alpha ,random_state=seed)

        elif stacking_models == 'Ridge':
            alpha = trial.suggest_float('alpha', 0.001,0.5, log=True)
            solver = trial.suggest_categorical('solver', ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga'])

            estimator = Ridge(alpha=alpha,solver=solver ,random_state=seed)
        
        
        # model = Pipeline([
            # ('preprocessor',preprocessor),
            # ('Stacking_estimator', estimator)
        # ])  

        cv_score = cross_validate(estimator, x_predicted, y_train, cv=KFold(n_splits=5,shuffle=True,random_state=seed), 
                                scoring='neg_mean_absolute_error',verbose=False,n_jobs=-1,return_train_score=True)
        
        trial.set_user_attr('train_cv',np.mean(cv_score['train_score']))
        trial.set_user_attr('val_cv',np.mean(cv_score['test_score']))
        
        mlflow.log_params(trial.params)
        mlflow.log_metric('train_cv_score',np.mean(cv_score['train_score']))
        mlflow.log_metric('val_cv_score',np.mean(cv_score['test_score']))
            
        return np.mean(cv_score['test_score'])


study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=seed),
                            study_name='StackingRegressor Hyperparameter Tuning')
    
with mlflow.start_run(run_name='With numerical cols') as parent :
    study.optimize(objective,n_trials=30,show_progress_bar=True)

    mlflow.log_metric('With numerical cols',study.best_value)
    mlflow.log_params(study.best_params)

    print('Best parameters:', study.best_params)
    print('Best score:', study.best_value)

[I 2026-06-22 21:55:23,400] A new study created in memory with name: StackingRegressor Hyperparameter Tuning


  0%|          | 0/30 [00:00<?, ?it/s]

🏃 View run trial_0 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/6/runs/378d2df58d1d4986a7aafc1af2aeee1d
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/6
[I 2026-06-22 21:55:28,442] Trial 0 finished with value: -3.05709812146938 and parameters: {'stacking_models': 'Lasso', 'alpha': 0.013913346327363716}. Best is trial 0 with value: -3.05709812146938.
🏃 View run trial_1 at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/6/runs/1cf7de149fec4493b54f94c1d99f601a
🧪 View experiment at: https://dagshub.com/shapniljoy/delivery-time-prediction.mlflow/#/experiments/6
[I 2026-06-22 21:55:31,641] Trial 1 finished with value: -3.1872021099877657 and parameters: {'stacking_models': 'DT', 'max_depth': 6, 'max_leaf_nodes': 9, 'min_samples_split': 13, 'min_samples_leaf': 8, 'max_features': 'sqrt'}. Best is trial 0 with value: -3.05709812146938.
🏃 View run trial_2 at: https://dagshub.com/

In [8]:
study.trials_dataframe()[['user_attrs_train_cv','user_attrs_val_cv','params_stacking_models']].sort_values(by='user_attrs_val_cv',ascending=False)

,user_attrs_train_cv,user_attrs_val_cv,params_stacking_models
26,-3.055944,-3.056723,Lasso
24,-3.055969,-3.056749,Lasso
23,-3.056009,-3.056789,Lasso
22,-3.056064,-3.056843,Lasso
14,-3.056074,-3.056853,Lasso
17,-3.056122,-3.056899,Lasso
16,-3.056150,-3.056926,Lasso
21,-3.056154,-3.056929,Lasso
18,-3.056197,-3.056971,Lasso
13,-3.056200,-3.056973,Lasso


Here, we have taken the out of box predicted values for each model and stacked them together to use as a input variables for the meta model hyperparamters tuning.  
The Optuna tuning resulted into a perfectly fitting model with lasso regression with no gap in the training and validation scores.  
Also it yielded the lowest test score among all the stacking models at MAE = 3.05 minutes.
We will choose this specification with lasso model as meta model to get our final predicted model.